In [ ]:
import os, random, warnings, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve, accuracy_score, f1_score
warnings.filterwarnings('ignore')

# ╔══════════════════════════════════════════════════════════╗
# ║  CONFIG                                                   ║
# ╚══════════════════════════════════════════════════════════╝
class CFG:
    DATA_ROOT     = '../input/breast-histopathology-images/BreaKHis_v1'
    OUT_DIR       = '/kaggle/working'
    BATCH_SIZE    = 32
    FOCAL_GAMMA   = 2.0
    FOCAL_ALPHA   = 0.33          # BreaKHis benign:malignant ~1:2
    LABEL_SMOOTH  = 0.05
    SEEDS         = [42, 123, 2024]   # <-- multi-seed pool; matches headline 40X/100X
    DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    AMP_ENABLED   = True
    TTA_FIVE_CROP = True
    # crude directional-flag thresholds (see docstring)
    FLAG_FLOOR    = 0.005         # ignore paired deltas smaller than half an AUC point

# Which magnifications THIS session runs. To split across sessions, set to ['40X'] or ['100X'].
MAGS_TO_RUN = ['40X']

# Per-magnification config (matches v0-10 headline hyperparameters)
MAG_CFG = {
    '40X':  dict(FREEZE_EPOCHS=15, UNFREEZE_EPOCHS=80,  DROPOUT_1=0.5, PATIENCE=35,
                 LR_P1=3e-4, LR_HEAD_P2=1e-4, LR_BACKBONE_P2=1e-5,
                 WEIGHT_DECAY=5e-3, P2_WARMUP=3, IMG_SIZE=224, AUG_STRENGTH='normal'),
    '100X': dict(FREEZE_EPOCHS=30, UNFREEZE_EPOCHS=90,  DROPOUT_1=0.5, PATIENCE=35,
                 LR_P1=3e-4, LR_HEAD_P2=1e-4, LR_BACKBONE_P2=1e-5,
                 WEIGHT_DECAY=5e-3, P2_WARMUP=3, IMG_SIZE=224, AUG_STRENGTH='normal'),
    '200X': dict(FREEZE_EPOCHS=12, UNFREEZE_EPOCHS=80,  DROPOUT_1=0.6, PATIENCE=35,
                 LR_P1=2e-4, LR_HEAD_P2=8e-5, LR_BACKBONE_P2=8e-6,
                 WEIGHT_DECAY=5e-3, P2_WARMUP=3, IMG_SIZE=224, AUG_STRENGTH='normal'),
    '400X': dict(FREEZE_EPOCHS=25, UNFREEZE_EPOCHS=100, DROPOUT_1=0.5, PATIENCE=15,
                 LR_P1=3e-4, LR_HEAD_P2=8e-5, LR_BACKBONE_P2=8e-6,
                 WEIGHT_DECAY=6e-3, P2_WARMUP=3, IMG_SIZE=320, AUG_STRENGTH='strong'),
}

ABLATIONS = ['full', 'no_mshra', 'no_spatial', 'no_channel', 'no_focal', 'no_twophase']

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.benchmark = True
os.makedirs(CFG.OUT_DIR, exist_ok=True)
RESULTS_JSON = os.path.join(CFG.OUT_DIR, 'ablation_40X_multiseed_results.json')
print(f"Device: {CFG.DEVICE}  |  Mags this session: {MAGS_TO_RUN}  |  Seeds: {CFG.SEEDS}")

# ╔══════════════════════════════════════════════════════════╗
# ║  RESUMABLE RESULT STORE                                   ║
# ║  key = f"{mag}|{seed}|{config}"  ->  metrics dict         ║
# ╚══════════════════════════════════════════════════════════╝
def load_store():
    if os.path.exists(RESULTS_JSON):
        with open(RESULTS_JSON) as f: return json.load(f)
    return {}
def save_store(store):
    tmp = RESULTS_JSON + '.tmp'
    with open(tmp, 'w') as f: json.dump(store, f, indent=2)
    os.replace(tmp, RESULTS_JSON)
STORE = load_store()
def skey(mag, seed, cfg): return f"{mag}|{seed}|{cfg}"

# ╔══════════════════════════════════════════════════════════╗
# ║  LOSS                                                     ║
# ╚══════════════════════════════════════════════════════════╝
def focal_loss(logits, labels):
    smooth = labels * (1 - CFG.LABEL_SMOOTH) + 0.5 * CFG.LABEL_SMOOTH
    bce = F.binary_cross_entropy_with_logits(logits, smooth, reduction='none')
    pt  = torch.where(labels == 1, torch.sigmoid(logits), 1 - torch.sigmoid(logits))
    a   = torch.where(labels == 1, torch.full_like(labels, CFG.FOCAL_ALPHA),
                      torch.full_like(labels, 1 - CFG.FOCAL_ALPHA))
    return (a * (1 - pt) ** CFG.FOCAL_GAMMA * bce).mean()

def plain_bce(logits, labels):
    return F.binary_cross_entropy_with_logits(logits, labels)

# ╔══════════════════════════════════════════════════════════╗
# ║  ATTENTION MODULES                                        ║
# ╚══════════════════════════════════════════════════════════╝
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False), nn.ReLU(inplace=True),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class MS_HRA_Block(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, use_channel=True, use_spatial=True):
        super().__init__()
        self.use_channel = use_channel
        self.use_spatial = use_spatial
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.ca    = ChannelAttention(out_ch) if use_channel else None
        self.sa    = SpatialAttention()       if use_spatial else None
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), nn.BatchNorm2d(out_ch))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.ca is not None: out = out * self.ca(out)
        if self.sa is not None: out = out * self.sa(out)
        return F.relu(out + self.shortcut(x))

# ╔══════════════════════════════════════════════════════════╗
# ║  MODEL (with ablation switch)                             ║
# ╚══════════════════════════════════════════════════════════╝
class Res_MSHRA(nn.Module):
    def __init__(self, dropout_1=0.5, ablation='full'):
        super().__init__()
        self.ablation = ablation
        base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.layer0 = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        if ablation == 'no_mshra':
            self.feature_stage = base.layer4   # genuine ResNet50 layer4: 1024->2048, stride 2
            self.is_mshra = False
        else:
            use_channel = (ablation != 'no_channel')
            use_spatial = (ablation != 'no_spatial')
            self.feature_stage = nn.Sequential(
                MS_HRA_Block(1024, 1024, stride=1, use_channel=use_channel, use_spatial=use_spatial),
                MS_HRA_Block(1024, 2048, stride=2, use_channel=use_channel, use_spatial=use_spatial))
            self.is_mshra = True
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(), nn.BatchNorm1d(2048), nn.Dropout(dropout_1),
            nn.Linear(2048, 512), nn.ReLU(inplace=True),
            nn.BatchNorm1d(512), nn.Dropout(0.3), nn.Linear(512, 1))
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.constant_(m.bias, 0)

    def backbone_params(self):
        p = (list(self.layer0.parameters()) + list(self.layer1.parameters()) +
             list(self.layer2.parameters()) + list(self.layer3.parameters()))
        if not self.is_mshra:
            p += list(self.feature_stage.parameters())
        return p
    def new_params(self):
        if self.is_mshra:
            return list(self.feature_stage.parameters()) + list(self.head.parameters())
        return list(self.head.parameters())
    def freeze_backbone(self):
        for p in self.backbone_params(): p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone_params(): p.requires_grad = True
    def forward(self, x):
        x = self.layer0(x); x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        x = self.feature_stage(x); x = self.gap(x)
        return self.head(x)

# ╔══════════════════════════════════════════════════════════╗
# ║  DATA                                                     ║
# ╚══════════════════════════════════════════════════════════╝
imagenet_norm = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

def get_train_transform(img_size, strength='normal'):
    if strength == 'strong':
        cj=(0.4,0.4,0.3,0.1); gray=0.10; erase_p=0.25; erase_sc=(0.02,0.15)
    else:
        cj=(0.2,0.2,0.15,0.05); gray=0.05; erase_p=0.15; erase_sc=(0.02,0.08)
    return transforms.Compose([
        transforms.Resize((int(img_size*1.15), int(img_size*1.15))),
        transforms.RandomCrop(img_size),
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=cj[0], contrast=cj[1], saturation=cj[2], hue=cj[3]),
        transforms.RandomGrayscale(p=gray),
        transforms.ToTensor(), imagenet_norm,
        transforms.RandomErasing(p=erase_p, scale=erase_sc)])

def get_val_transform(img_size):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)), transforms.ToTensor(), imagenet_norm])

class HistologyDataset(Dataset):
    def __init__(self, mag, split, transform=None):
        self.samples = []; self.transform = transform
        data_root = CFG.DATA_ROOT
        for root, dirs, _ in os.walk(data_root):
            if mag in dirs:
                data_root = root; break
        for cls, lbl in [('benign', 0), ('malignant', 1)]:
            path = os.path.join(data_root, mag, split, cls)
            if os.path.exists(path):
                for f in sorted(os.listdir(path)):
                    if f.lower().endswith(('.png','.jpg','.jpeg')):
                        self.samples.append((os.path.join(path, f), lbl))
        n_b = sum(1 for _,l in self.samples if l==0)
        n_m = sum(1 for _,l in self.samples if l==1)
        print(f"    [{mag}/{split}] benign={n_b} malignant={n_m} total={len(self.samples)}")
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, lbl = self.samples[idx]
        img = Image.open(path).convert('RGB')
        t = self.transform(img) if self.transform else transforms.ToTensor()(img)
        return t, torch.tensor(lbl, dtype=torch.float32)
    def get_labels(self): return [l for _,l in self.samples]

def make_train_loader(ds, seed):
    labels = np.array(ds.get_labels())
    counts = np.maximum(np.bincount(labels.astype(int), minlength=2), 1)
    w = (1.0 / counts)[labels.astype(int)]
    g = torch.Generator(); g.manual_seed(seed)   # seed the sampler too
    sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), len(ds),
                                    replacement=True, generator=g)
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                      num_workers=4, pin_memory=True, drop_last=True)

def make_val_loader(ds):
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                      num_workers=4, pin_memory=True, drop_last=False)

# ╔══════════════════════════════════════════════════════════╗
# ║  TRAIN / EVAL                                             ║
# ╚══════════════════════════════════════════════════════════╝
def train_one_epoch(model, loader, opt, scaler, loss_fn):
    model.train()
    for imgs, labels in loader:
        imgs=imgs.to(CFG.DEVICE,non_blocking=True); labels=labels.to(CFG.DEVICE,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        if CFG.AMP_ENABLED:
            with torch.cuda.amp.autocast():
                logits = model(imgs).squeeze(1); loss = loss_fn(logits, labels)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        else:
            logits = model(imgs).squeeze(1); loss = loss_fn(logits, labels)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()

@torch.no_grad()
def eval_auc(model, loader):
    model.eval(); probs=[]; tgts=[]
    for imgs, labels in loader:
        if CFG.AMP_ENABLED:
            with torch.cuda.amp.autocast():
                lo = model(imgs.to(CFG.DEVICE,non_blocking=True)).squeeze(1)
        else:
            lo = model(imgs.to(CFG.DEVICE,non_blocking=True)).squeeze(1)
        probs.extend(torch.sigmoid(lo.float()).cpu().numpy()); tgts.extend(labels.numpy())
    return roc_auc_score(tgts, probs)

@torch.no_grad()
def predict(model, loader, img_size, use_tta):
    model.eval(); P=[]; Y=[]
    crop = int(img_size*0.9)
    for imgs, labels in loader:
        imgs = imgs.to(CFG.DEVICE, non_blocking=True)
        if use_tta:
            views = [imgs, torch.flip(imgs,[3]), torch.flip(imgs,[2]), torch.flip(imgs,[2,3]),
                     torch.rot90(imgs,1,[2,3]), torch.rot90(imgs,3,[2,3])]
            if CFG.TTA_FIVE_CROP:
                H,W=imgs.shape[2],imgs.shape[3]
                offs=[((H-crop)//2,(W-crop)//2),(0,0),(0,W-crop),(H-crop,0),(H-crop,W-crop)]
                for t,l in offs:
                    c=imgs[:,:,t:t+crop,l:l+crop]
                    c=F.interpolate(c,size=(H,W),mode='bilinear',align_corners=False); views.append(c)
            if CFG.AMP_ENABLED:
                with torch.cuda.amp.autocast():
                    pr = sum(torch.sigmoid(model(v).float()) for v in views)/len(views)
            else:
                pr = sum(torch.sigmoid(model(v)) for v in views)/len(views)
        else:
            if CFG.AMP_ENABLED:
                with torch.cuda.amp.autocast():
                    pr = torch.sigmoid(model(imgs).float())
            else:
                pr = torch.sigmoid(model(imgs))
        P.extend(pr.squeeze(1).cpu().numpy()); Y.extend(labels.numpy())
    return np.array(P), np.array(Y)

def metrics_at_best_threshold(y_true, y_prob):
    fpr,tpr,thr = roc_curve(y_true,y_prob)
    accs=[accuracy_score(y_true,(y_prob>=t).astype(int)) for t in thr]
    bt=thr[int(np.argmax(accs))]
    y_pred=(y_prob>=bt).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_true,y_pred).ravel()
    sens=tp/(tp+fn) if (tp+fn)>0 else 0
    spec=tn/(tn+fp) if (tn+fp)>0 else 0
    return dict(auc=float(roc_auc_score(y_true,y_prob)), acc=float(max(accs)),
                sens=float(sens), spec=float(spec), f1=float(f1_score(y_true,y_pred)))

# ╔══════════════════════════════════════════════════════════╗
# ║  ONE (mag, seed, config) RUN                              ║
# ╚══════════════════════════════════════════════════════════╝
def run_one(mag, seed, ablation, train_loader, val_loader):
    mc = MAG_CFG[mag]; img_size = mc['IMG_SIZE']
    set_seed(seed)                              # per-seed, not fixed
    loss_fn = plain_bce if ablation == 'no_focal' else focal_loss
    two_phase = (ablation != 'no_twophase')

    model = Res_MSHRA(dropout_1=mc['DROPOUT_1'], ablation=ablation).to(CFG.DEVICE)
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.AMP_ENABLED)
    best_auc = 0.0
    ckpt = os.path.join(CFG.OUT_DIR, f'abl_{mag}_s{seed}_{ablation}.pth')

    if two_phase:
        model.freeze_backbone()
        opt = torch.optim.AdamW(model.new_params(), lr=mc['LR_P1'], weight_decay=mc['WEIGHT_DECAY'])
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=mc['FREEZE_EPOCHS'], eta_min=1e-6)
        for ep in range(1, mc['FREEZE_EPOCHS']+1):
            train_one_epoch(model, train_loader, opt, scaler, loss_fn)
            va = eval_auc(model, val_loader); sched.step()
            if va > best_auc: best_auc=va; torch.save(model.state_dict(), ckpt)
        model.unfreeze_backbone()
        opt = torch.optim.AdamW([
            {'params': model.backbone_params(), 'lr': mc['LR_BACKBONE_P2']},
            {'params': model.new_params(),      'lr': mc['LR_HEAD_P2']}],
            weight_decay=mc['WEIGHT_DECAY'])
        warm=mc['P2_WARMUP']; cos=mc['UNFREEZE_EPOCHS']-warm
        sched = torch.optim.lr_scheduler.SequentialLR(opt, schedulers=[
            torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=warm),
            torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cos, eta_min=1e-7)],
            milestones=[warm])
        patience=mc['PATIENCE']; ctr=0
        for ep in range(1, mc['UNFREEZE_EPOCHS']+1):
            train_one_epoch(model, train_loader, opt, scaler, loss_fn)
            va = eval_auc(model, val_loader); sched.step()
            if va > best_auc: best_auc=va; ctr=0; torch.save(model.state_dict(), ckpt)
            else: ctr += 1
            if ctr >= patience: break
    else:
        model.unfreeze_backbone()
        total_ep = mc['FREEZE_EPOCHS'] + mc['UNFREEZE_EPOCHS']
        opt = torch.optim.AdamW([
            {'params': model.backbone_params(), 'lr': mc['LR_BACKBONE_P2']},
            {'params': model.new_params(),      'lr': mc['LR_HEAD_P2']}],
            weight_decay=mc['WEIGHT_DECAY'])
        warm=mc['P2_WARMUP']; cos=total_ep-warm
        sched = torch.optim.lr_scheduler.SequentialLR(opt, schedulers=[
            torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=warm),
            torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cos, eta_min=1e-7)],
            milestones=[warm])
        patience=mc['PATIENCE']; ctr=0
        for ep in range(1, total_ep+1):
            train_one_epoch(model, train_loader, opt, scaler, loss_fn)
            va = eval_auc(model, val_loader); sched.step()
            if va > best_auc: best_auc=va; ctr=0; torch.save(model.state_dict(), ckpt)
            else: ctr += 1
            if ctr >= patience: break

    model.load_state_dict(torch.load(ckpt, map_location=CFG.DEVICE))
    yP, yT = predict(model, val_loader, img_size, use_tta=True)
    m_tta = metrics_at_best_threshold(yT, yP)
    m_notta = None
    if ablation == 'full':
        yP2, yT2 = predict(model, val_loader, img_size, use_tta=False)
        m_notta = metrics_at_best_threshold(yT2, yP2)
    try: os.remove(ckpt)                         # keep /kaggle/working small
    except OSError: pass
    return m_tta, m_notta

# ╔══════════════════════════════════════════════════════════╗
# ║  RUN: loop mag -> seed -> config (resumable)              ║
# ╚══════════════════════════════════════════════════════════╝
overall_t0 = time.time()
for mag in MAGS_TO_RUN:
    print(f"\n{'='*64}\n  MAGNIFICATION {mag}\n{'='*64}")
    mc = MAG_CFG[mag]; img_size = mc['IMG_SIZE']
    for seed in CFG.SEEDS:
        print(f"\n  ----- {mag} | seed {seed} -----")
        tr_ds = HistologyDataset(mag, 'train', get_train_transform(img_size, mc['AUG_STRENGTH']))
        va_ds = HistologyDataset(mag, 'test',  get_val_transform(img_size))
        tr_loader = make_train_loader(tr_ds, seed); va_loader = make_val_loader(va_ds)
        for ab in ABLATIONS:
            if skey(mag, seed, ab) in STORE and (ab != 'full' or skey(mag, seed, 'full_no_tta') in STORE):
                m = STORE[skey(mag, seed, ab)]
                print(f"    [skip] {ab:12s} AUC={m['auc']:.4f} (cached)")
                continue
            t0 = time.time()
            m_tta, m_notta = run_one(mag, seed, ab, tr_loader, va_loader)
            STORE[skey(mag, seed, ab)] = m_tta
            if ab == 'full' and m_notta is not None:
                STORE[skey(mag, seed, 'full_no_tta')] = m_notta
            save_store(STORE)
            print(f"    {ab:12s} AUC={m_tta['auc']:.4f} Acc={m_tta['acc']*100:.2f}% "
                  f"Sens={m_tta['sens']:.3f} Spec={m_tta['spec']:.3f} F1={m_tta['f1']:.3f}  "
                  f"({(time.time()-t0)/60:.1f} min)")
            if ab == 'full' and m_notta is not None:
                print(f"    {'full(noTTA)':12s} AUC={m_notta['auc']:.4f} (free)")

# ╔══════════════════════════════════════════════════════════╗
# ║  AGGREGATE: mean +/- std + PAIRED dAUC across seeds       ║
# ╚══════════════════════════════════════════════════════════╝
ROW_ORDER = ['full', 'full_no_tta', 'no_mshra', 'no_spatial', 'no_channel', 'no_focal', 'no_twophase']
ROW_LABEL = {
    'full':        'Full Res-MSHRA',
    'full_no_tta': '  - TTA (full model, no TTA)',
    'no_mshra':    '  - MS-HRA blocks (plain ResNet50)',
    'no_spatial':  '  - Spatial attention',
    'no_channel':  '  - Channel attention',
    'no_focal':    '  - Focal loss (plain BCE)',
    'no_twophase': '  - Two-phase training (end-to-end)',
}

def collect(mag, cfg, metric='auc'):
    vals=[]
    for seed in CFG.SEEDS:
        k=skey(mag,seed,cfg)
        if k in STORE: vals.append(STORE[k][metric])
    return np.array(vals, dtype=float)

print(f"\n\n{'#'*82}")
print(f"  MULTI-SEED ABLATION — 40X only  seeds={CFG.SEEDS}")
print(f"  paired dAUC = mean_seed( AUC(config) - AUC(full) ).  negative => component HELPED.")
print(f"  flag '*' = |mean dAUC| >= std dAUC AND |mean dAUC| >= {CFG.FLAG_FLOOR} (crude, not a p-value)")
print(f"{'#'*82}")

for mag in MAGS_TO_RUN:
    full_aucs = collect(mag, 'full')
    if len(full_aucs)==0:
        print(f"\n=== {mag} === (no results yet)"); continue
    n = len(full_aucs)
    print(f"\n=== {mag} ===   (n={n} seeds; full AUC = {full_aucs.mean():.4f} +/- {full_aucs.std(ddof=1) if n>1 else 0:.4f})")
    print(f"  {'Configuration':<36s} {'mean AUC':>10s} {'std':>7s} {'paired dAUC':>18s}  flag")
    print(f"  {'-'*80}")
    for key in ROW_ORDER:
        aucs = collect(mag, key)
        if len(aucs)==0: continue
        mu = aucs.mean(); sd = aucs.std(ddof=1) if len(aucs)>1 else 0.0
        if key == 'full':
            print(f"  {ROW_LABEL[key]:<36s} {mu:>10.4f} {sd:>7.4f} {'(reference)':>18s}")
            continue
        # paired delta: align seed-by-seed against full (or full_no_tta's own full)
        deltas=[]
        for seed in CFG.SEEDS:
            kc=skey(mag,seed,key); kf=skey(mag,seed,'full')
            if kc in STORE and kf in STORE:
                deltas.append(STORE[kc]['auc'] - STORE[kf]['auc'])
        deltas=np.array(deltas, dtype=float)
        if len(deltas)>0:
            dmu=deltas.mean(); dsd=deltas.std(ddof=1) if len(deltas)>1 else 0.0
            flag = '*' if (abs(dmu) >= dsd and abs(dmu) >= CFG.FLAG_FLOOR) else ''
            dstr=f"{dmu:+.4f} +/- {dsd:.4f}"
        else:
            dstr="n/a"; flag=''
        print(f"  {ROW_LABEL[key]:<36s} {mu:>10.4f} {sd:>7.4f} {dstr:>18s}   {flag}")

print(f"\n  Total wall time this session: {(time.time()-overall_t0)/60:.1f} min")
print(f"  Results JSON (resumable): {RESULTS_JSON}")
print(f"\n  Copy this entire output (and download the 40X JSON) and send it back.")


In [ ]:
import os, random, warnings, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve, accuracy_score, f1_score
warnings.filterwarnings('ignore')

# ╔══════════════════════════════════════════════════════════╗
# ║  CONFIG                                                   ║
# ╚══════════════════════════════════════════════════════════╝
class CFG:
    DATA_ROOT     = '../input/breast-histopathology-images/BreaKHis_v1'
    OUT_DIR       = '/kaggle/working'
    BATCH_SIZE    = 32
    FOCAL_GAMMA   = 2.0
    FOCAL_ALPHA   = 0.33          # BreaKHis benign:malignant ~1:2
    LABEL_SMOOTH  = 0.05
    SEEDS         = [42, 123, 2024]   # <-- multi-seed pool; matches headline 40X/100X
    DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    AMP_ENABLED   = True
    TTA_FIVE_CROP = True
    # crude directional-flag thresholds (see docstring)
    FLAG_FLOOR    = 0.005         # ignore paired deltas smaller than half an AUC point

# Which magnifications THIS session runs. To split across sessions, set to ['40X'] or ['100X'].
MAGS_TO_RUN = ['100X']

# Per-magnification config (matches v0-10 headline hyperparameters)
MAG_CFG = {
    '40X':  dict(FREEZE_EPOCHS=15, UNFREEZE_EPOCHS=80,  DROPOUT_1=0.5, PATIENCE=35,
                 LR_P1=3e-4, LR_HEAD_P2=1e-4, LR_BACKBONE_P2=1e-5,
                 WEIGHT_DECAY=5e-3, P2_WARMUP=3, IMG_SIZE=224, AUG_STRENGTH='normal'),
    '100X': dict(FREEZE_EPOCHS=30, UNFREEZE_EPOCHS=90,  DROPOUT_1=0.5, PATIENCE=35,
                 LR_P1=3e-4, LR_HEAD_P2=1e-4, LR_BACKBONE_P2=1e-5,
                 WEIGHT_DECAY=5e-3, P2_WARMUP=3, IMG_SIZE=224, AUG_STRENGTH='normal'),
    '200X': dict(FREEZE_EPOCHS=12, UNFREEZE_EPOCHS=80,  DROPOUT_1=0.6, PATIENCE=35,
                 LR_P1=2e-4, LR_HEAD_P2=8e-5, LR_BACKBONE_P2=8e-6,
                 WEIGHT_DECAY=5e-3, P2_WARMUP=3, IMG_SIZE=224, AUG_STRENGTH='normal'),
    '400X': dict(FREEZE_EPOCHS=25, UNFREEZE_EPOCHS=100, DROPOUT_1=0.5, PATIENCE=15,
                 LR_P1=3e-4, LR_HEAD_P2=8e-5, LR_BACKBONE_P2=8e-6,
                 WEIGHT_DECAY=6e-3, P2_WARMUP=3, IMG_SIZE=320, AUG_STRENGTH='strong'),
}

ABLATIONS = ['full', 'no_mshra', 'no_spatial', 'no_channel', 'no_focal', 'no_twophase']

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.benchmark = True
os.makedirs(CFG.OUT_DIR, exist_ok=True)
RESULTS_JSON = os.path.join(CFG.OUT_DIR, 'ablation_100X_multiseed_results.json')
print(f"Device: {CFG.DEVICE}  |  Mags this session: {MAGS_TO_RUN}  |  Seeds: {CFG.SEEDS}")

# ╔══════════════════════════════════════════════════════════╗
# ║  RESUMABLE RESULT STORE                                   ║
# ║  key = f"{mag}|{seed}|{config}"  ->  metrics dict         ║
# ╚══════════════════════════════════════════════════════════╝
def load_store():
    if os.path.exists(RESULTS_JSON):
        with open(RESULTS_JSON) as f: return json.load(f)
    return {}
def save_store(store):
    tmp = RESULTS_JSON + '.tmp'
    with open(tmp, 'w') as f: json.dump(store, f, indent=2)
    os.replace(tmp, RESULTS_JSON)
STORE = load_store()
def skey(mag, seed, cfg): return f"{mag}|{seed}|{cfg}"

# ╔══════════════════════════════════════════════════════════╗
# ║  LOSS                                                     ║
# ╚══════════════════════════════════════════════════════════╝
def focal_loss(logits, labels):
    smooth = labels * (1 - CFG.LABEL_SMOOTH) + 0.5 * CFG.LABEL_SMOOTH
    bce = F.binary_cross_entropy_with_logits(logits, smooth, reduction='none')
    pt  = torch.where(labels == 1, torch.sigmoid(logits), 1 - torch.sigmoid(logits))
    a   = torch.where(labels == 1, torch.full_like(labels, CFG.FOCAL_ALPHA),
                      torch.full_like(labels, 1 - CFG.FOCAL_ALPHA))
    return (a * (1 - pt) ** CFG.FOCAL_GAMMA * bce).mean()

def plain_bce(logits, labels):
    return F.binary_cross_entropy_with_logits(logits, labels)

# ╔══════════════════════════════════════════════════════════╗
# ║  ATTENTION MODULES                                        ║
# ╚══════════════════════════════════════════════════════════╝
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False), nn.ReLU(inplace=True),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class MS_HRA_Block(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, use_channel=True, use_spatial=True):
        super().__init__()
        self.use_channel = use_channel
        self.use_spatial = use_spatial
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.ca    = ChannelAttention(out_ch) if use_channel else None
        self.sa    = SpatialAttention()       if use_spatial else None
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), nn.BatchNorm2d(out_ch))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.ca is not None: out = out * self.ca(out)
        if self.sa is not None: out = out * self.sa(out)
        return F.relu(out + self.shortcut(x))

# ╔══════════════════════════════════════════════════════════╗
# ║  MODEL (with ablation switch)                             ║
# ╚══════════════════════════════════════════════════════════╝
class Res_MSHRA(nn.Module):
    def __init__(self, dropout_1=0.5, ablation='full'):
        super().__init__()
        self.ablation = ablation
        base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.layer0 = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        if ablation == 'no_mshra':
            self.feature_stage = base.layer4   # genuine ResNet50 layer4: 1024->2048, stride 2
            self.is_mshra = False
        else:
            use_channel = (ablation != 'no_channel')
            use_spatial = (ablation != 'no_spatial')
            self.feature_stage = nn.Sequential(
                MS_HRA_Block(1024, 1024, stride=1, use_channel=use_channel, use_spatial=use_spatial),
                MS_HRA_Block(1024, 2048, stride=2, use_channel=use_channel, use_spatial=use_spatial))
            self.is_mshra = True
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(), nn.BatchNorm1d(2048), nn.Dropout(dropout_1),
            nn.Linear(2048, 512), nn.ReLU(inplace=True),
            nn.BatchNorm1d(512), nn.Dropout(0.3), nn.Linear(512, 1))
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.constant_(m.bias, 0)

    def backbone_params(self):
        p = (list(self.layer0.parameters()) + list(self.layer1.parameters()) +
             list(self.layer2.parameters()) + list(self.layer3.parameters()))
        if not self.is_mshra:
            p += list(self.feature_stage.parameters())
        return p
    def new_params(self):
        if self.is_mshra:
            return list(self.feature_stage.parameters()) + list(self.head.parameters())
        return list(self.head.parameters())
    def freeze_backbone(self):
        for p in self.backbone_params(): p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone_params(): p.requires_grad = True
    def forward(self, x):
        x = self.layer0(x); x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        x = self.feature_stage(x); x = self.gap(x)
        return self.head(x)

# ╔══════════════════════════════════════════════════════════╗
# ║  DATA                                                     ║
# ╚══════════════════════════════════════════════════════════╝
imagenet_norm = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

def get_train_transform(img_size, strength='normal'):
    if strength == 'strong':
        cj=(0.4,0.4,0.3,0.1); gray=0.10; erase_p=0.25; erase_sc=(0.02,0.15)
    else:
        cj=(0.2,0.2,0.15,0.05); gray=0.05; erase_p=0.15; erase_sc=(0.02,0.08)
    return transforms.Compose([
        transforms.Resize((int(img_size*1.15), int(img_size*1.15))),
        transforms.RandomCrop(img_size),
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=cj[0], contrast=cj[1], saturation=cj[2], hue=cj[3]),
        transforms.RandomGrayscale(p=gray),
        transforms.ToTensor(), imagenet_norm,
        transforms.RandomErasing(p=erase_p, scale=erase_sc)])

def get_val_transform(img_size):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)), transforms.ToTensor(), imagenet_norm])

class HistologyDataset(Dataset):
    def __init__(self, mag, split, transform=None):
        self.samples = []; self.transform = transform
        data_root = CFG.DATA_ROOT
        for root, dirs, _ in os.walk(data_root):
            if mag in dirs:
                data_root = root; break
        for cls, lbl in [('benign', 0), ('malignant', 1)]:
            path = os.path.join(data_root, mag, split, cls)
            if os.path.exists(path):
                for f in sorted(os.listdir(path)):
                    if f.lower().endswith(('.png','.jpg','.jpeg')):
                        self.samples.append((os.path.join(path, f), lbl))
        n_b = sum(1 for _,l in self.samples if l==0)
        n_m = sum(1 for _,l in self.samples if l==1)
        print(f"    [{mag}/{split}] benign={n_b} malignant={n_m} total={len(self.samples)}")
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, lbl = self.samples[idx]
        img = Image.open(path).convert('RGB')
        t = self.transform(img) if self.transform else transforms.ToTensor()(img)
        return t, torch.tensor(lbl, dtype=torch.float32)
    def get_labels(self): return [l for _,l in self.samples]

def make_train_loader(ds, seed):
    labels = np.array(ds.get_labels())
    counts = np.maximum(np.bincount(labels.astype(int), minlength=2), 1)
    w = (1.0 / counts)[labels.astype(int)]
    g = torch.Generator(); g.manual_seed(seed)   # seed the sampler too
    sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), len(ds),
                                    replacement=True, generator=g)
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                      num_workers=4, pin_memory=True, drop_last=True)

def make_val_loader(ds):
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                      num_workers=4, pin_memory=True, drop_last=False)

# ╔══════════════════════════════════════════════════════════╗
# ║  TRAIN / EVAL                                             ║
# ╚══════════════════════════════════════════════════════════╝
def train_one_epoch(model, loader, opt, scaler, loss_fn):
    model.train()
    for imgs, labels in loader:
        imgs=imgs.to(CFG.DEVICE,non_blocking=True); labels=labels.to(CFG.DEVICE,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        if CFG.AMP_ENABLED:
            with torch.cuda.amp.autocast():
                logits = model(imgs).squeeze(1); loss = loss_fn(logits, labels)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        else:
            logits = model(imgs).squeeze(1); loss = loss_fn(logits, labels)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()

@torch.no_grad()
def eval_auc(model, loader):
    model.eval(); probs=[]; tgts=[]
    for imgs, labels in loader:
        if CFG.AMP_ENABLED:
            with torch.cuda.amp.autocast():
                lo = model(imgs.to(CFG.DEVICE,non_blocking=True)).squeeze(1)
        else:
            lo = model(imgs.to(CFG.DEVICE,non_blocking=True)).squeeze(1)
        probs.extend(torch.sigmoid(lo.float()).cpu().numpy()); tgts.extend(labels.numpy())
    return roc_auc_score(tgts, probs)

@torch.no_grad()
def predict(model, loader, img_size, use_tta):
    model.eval(); P=[]; Y=[]
    crop = int(img_size*0.9)
    for imgs, labels in loader:
        imgs = imgs.to(CFG.DEVICE, non_blocking=True)
        if use_tta:
            views = [imgs, torch.flip(imgs,[3]), torch.flip(imgs,[2]), torch.flip(imgs,[2,3]),
                     torch.rot90(imgs,1,[2,3]), torch.rot90(imgs,3,[2,3])]
            if CFG.TTA_FIVE_CROP:
                H,W=imgs.shape[2],imgs.shape[3]
                offs=[((H-crop)//2,(W-crop)//2),(0,0),(0,W-crop),(H-crop,0),(H-crop,W-crop)]
                for t,l in offs:
                    c=imgs[:,:,t:t+crop,l:l+crop]
                    c=F.interpolate(c,size=(H,W),mode='bilinear',align_corners=False); views.append(c)
            if CFG.AMP_ENABLED:
                with torch.cuda.amp.autocast():
                    pr = sum(torch.sigmoid(model(v).float()) for v in views)/len(views)
            else:
                pr = sum(torch.sigmoid(model(v)) for v in views)/len(views)
        else:
            if CFG.AMP_ENABLED:
                with torch.cuda.amp.autocast():
                    pr = torch.sigmoid(model(imgs).float())
            else:
                pr = torch.sigmoid(model(imgs))
        P.extend(pr.squeeze(1).cpu().numpy()); Y.extend(labels.numpy())
    return np.array(P), np.array(Y)

def metrics_at_best_threshold(y_true, y_prob):
    fpr,tpr,thr = roc_curve(y_true,y_prob)
    accs=[accuracy_score(y_true,(y_prob>=t).astype(int)) for t in thr]
    bt=thr[int(np.argmax(accs))]
    y_pred=(y_prob>=bt).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_true,y_pred).ravel()
    sens=tp/(tp+fn) if (tp+fn)>0 else 0
    spec=tn/(tn+fp) if (tn+fp)>0 else 0
    return dict(auc=float(roc_auc_score(y_true,y_prob)), acc=float(max(accs)),
                sens=float(sens), spec=float(spec), f1=float(f1_score(y_true,y_pred)))

# ╔══════════════════════════════════════════════════════════╗
# ║  ONE (mag, seed, config) RUN                              ║
# ╚══════════════════════════════════════════════════════════╝
def run_one(mag, seed, ablation, train_loader, val_loader):
    mc = MAG_CFG[mag]; img_size = mc['IMG_SIZE']
    set_seed(seed)                              # per-seed, not fixed
    loss_fn = plain_bce if ablation == 'no_focal' else focal_loss
    two_phase = (ablation != 'no_twophase')

    model = Res_MSHRA(dropout_1=mc['DROPOUT_1'], ablation=ablation).to(CFG.DEVICE)
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.AMP_ENABLED)
    best_auc = 0.0
    ckpt = os.path.join(CFG.OUT_DIR, f'abl_{mag}_s{seed}_{ablation}.pth')

    if two_phase:
        model.freeze_backbone()
        opt = torch.optim.AdamW(model.new_params(), lr=mc['LR_P1'], weight_decay=mc['WEIGHT_DECAY'])
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=mc['FREEZE_EPOCHS'], eta_min=1e-6)
        for ep in range(1, mc['FREEZE_EPOCHS']+1):
            train_one_epoch(model, train_loader, opt, scaler, loss_fn)
            va = eval_auc(model, val_loader); sched.step()
            if va > best_auc: best_auc=va; torch.save(model.state_dict(), ckpt)
        model.unfreeze_backbone()
        opt = torch.optim.AdamW([
            {'params': model.backbone_params(), 'lr': mc['LR_BACKBONE_P2']},
            {'params': model.new_params(),      'lr': mc['LR_HEAD_P2']}],
            weight_decay=mc['WEIGHT_DECAY'])
        warm=mc['P2_WARMUP']; cos=mc['UNFREEZE_EPOCHS']-warm
        sched = torch.optim.lr_scheduler.SequentialLR(opt, schedulers=[
            torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=warm),
            torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cos, eta_min=1e-7)],
            milestones=[warm])
        patience=mc['PATIENCE']; ctr=0
        for ep in range(1, mc['UNFREEZE_EPOCHS']+1):
            train_one_epoch(model, train_loader, opt, scaler, loss_fn)
            va = eval_auc(model, val_loader); sched.step()
            if va > best_auc: best_auc=va; ctr=0; torch.save(model.state_dict(), ckpt)
            else: ctr += 1
            if ctr >= patience: break
    else:
        model.unfreeze_backbone()
        total_ep = mc['FREEZE_EPOCHS'] + mc['UNFREEZE_EPOCHS']
        opt = torch.optim.AdamW([
            {'params': model.backbone_params(), 'lr': mc['LR_BACKBONE_P2']},
            {'params': model.new_params(),      'lr': mc['LR_HEAD_P2']}],
            weight_decay=mc['WEIGHT_DECAY'])
        warm=mc['P2_WARMUP']; cos=total_ep-warm
        sched = torch.optim.lr_scheduler.SequentialLR(opt, schedulers=[
            torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=warm),
            torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cos, eta_min=1e-7)],
            milestones=[warm])
        patience=mc['PATIENCE']; ctr=0
        for ep in range(1, total_ep+1):
            train_one_epoch(model, train_loader, opt, scaler, loss_fn)
            va = eval_auc(model, val_loader); sched.step()
            if va > best_auc: best_auc=va; ctr=0; torch.save(model.state_dict(), ckpt)
            else: ctr += 1
            if ctr >= patience: break

    model.load_state_dict(torch.load(ckpt, map_location=CFG.DEVICE))
    yP, yT = predict(model, val_loader, img_size, use_tta=True)
    m_tta = metrics_at_best_threshold(yT, yP)
    m_notta = None
    if ablation == 'full':
        yP2, yT2 = predict(model, val_loader, img_size, use_tta=False)
        m_notta = metrics_at_best_threshold(yT2, yP2)
    try: os.remove(ckpt)                         # keep /kaggle/working small
    except OSError: pass
    return m_tta, m_notta

# ╔══════════════════════════════════════════════════════════╗
# ║  RUN: loop mag -> seed -> config (resumable)              ║
# ╚══════════════════════════════════════════════════════════╝
overall_t0 = time.time()
for mag in MAGS_TO_RUN:
    print(f"\n{'='*64}\n  MAGNIFICATION {mag}\n{'='*64}")
    mc = MAG_CFG[mag]; img_size = mc['IMG_SIZE']
    for seed in CFG.SEEDS:
        print(f"\n  ----- {mag} | seed {seed} -----")
        tr_ds = HistologyDataset(mag, 'train', get_train_transform(img_size, mc['AUG_STRENGTH']))
        va_ds = HistologyDataset(mag, 'test',  get_val_transform(img_size))
        tr_loader = make_train_loader(tr_ds, seed); va_loader = make_val_loader(va_ds)
        for ab in ABLATIONS:
            if skey(mag, seed, ab) in STORE and (ab != 'full' or skey(mag, seed, 'full_no_tta') in STORE):
                m = STORE[skey(mag, seed, ab)]
                print(f"    [skip] {ab:12s} AUC={m['auc']:.4f} (cached)")
                continue
            t0 = time.time()
            m_tta, m_notta = run_one(mag, seed, ab, tr_loader, va_loader)
            STORE[skey(mag, seed, ab)] = m_tta
            if ab == 'full' and m_notta is not None:
                STORE[skey(mag, seed, 'full_no_tta')] = m_notta
            save_store(STORE)
            print(f"    {ab:12s} AUC={m_tta['auc']:.4f} Acc={m_tta['acc']*100:.2f}% "
                  f"Sens={m_tta['sens']:.3f} Spec={m_tta['spec']:.3f} F1={m_tta['f1']:.3f}  "
                  f"({(time.time()-t0)/60:.1f} min)")
            if ab == 'full' and m_notta is not None:
                print(f"    {'full(noTTA)':12s} AUC={m_notta['auc']:.4f} (free)")

# ╔══════════════════════════════════════════════════════════╗
# ║  AGGREGATE: mean +/- std + PAIRED dAUC across seeds       ║
# ╚══════════════════════════════════════════════════════════╝
ROW_ORDER = ['full', 'full_no_tta', 'no_mshra', 'no_spatial', 'no_channel', 'no_focal', 'no_twophase']
ROW_LABEL = {
    'full':        'Full Res-MSHRA',
    'full_no_tta': '  - TTA (full model, no TTA)',
    'no_mshra':    '  - MS-HRA blocks (plain ResNet50)',
    'no_spatial':  '  - Spatial attention',
    'no_channel':  '  - Channel attention',
    'no_focal':    '  - Focal loss (plain BCE)',
    'no_twophase': '  - Two-phase training (end-to-end)',
}

def collect(mag, cfg, metric='auc'):
    vals=[]
    for seed in CFG.SEEDS:
        k=skey(mag,seed,cfg)
        if k in STORE: vals.append(STORE[k][metric])
    return np.array(vals, dtype=float)

print(f"\n\n{'#'*82}")
print(f"  MULTI-SEED ABLATION — 100X only  seeds={CFG.SEEDS}")
print(f"  paired dAUC = mean_seed( AUC(config) - AUC(full) ).  negative => component HELPED.")
print(f"  flag '*' = |mean dAUC| >= std dAUC AND |mean dAUC| >= {CFG.FLAG_FLOOR} (crude, not a p-value)")
print(f"{'#'*82}")

for mag in MAGS_TO_RUN:
    full_aucs = collect(mag, 'full')
    if len(full_aucs)==0:
        print(f"\n=== {mag} === (no results yet)"); continue
    n = len(full_aucs)
    print(f"\n=== {mag} ===   (n={n} seeds; full AUC = {full_aucs.mean():.4f} +/- {full_aucs.std(ddof=1) if n>1 else 0:.4f})")
    print(f"  {'Configuration':<36s} {'mean AUC':>10s} {'std':>7s} {'paired dAUC':>18s}  flag")
    print(f"  {'-'*80}")
    for key in ROW_ORDER:
        aucs = collect(mag, key)
        if len(aucs)==0: continue
        mu = aucs.mean(); sd = aucs.std(ddof=1) if len(aucs)>1 else 0.0
        if key == 'full':
            print(f"  {ROW_LABEL[key]:<36s} {mu:>10.4f} {sd:>7.4f} {'(reference)':>18s}")
            continue
        # paired delta: align seed-by-seed against full (or full_no_tta's own full)
        deltas=[]
        for seed in CFG.SEEDS:
            kc=skey(mag,seed,key); kf=skey(mag,seed,'full')
            if kc in STORE and kf in STORE:
                deltas.append(STORE[kc]['auc'] - STORE[kf]['auc'])
        deltas=np.array(deltas, dtype=float)
        if len(deltas)>0:
            dmu=deltas.mean(); dsd=deltas.std(ddof=1) if len(deltas)>1 else 0.0
            flag = '*' if (abs(dmu) >= dsd and abs(dmu) >= CFG.FLAG_FLOOR) else ''
            dstr=f"{dmu:+.4f} +/- {dsd:.4f}"
        else:
            dstr="n/a"; flag=''
        print(f"  {ROW_LABEL[key]:<36s} {mu:>10.4f} {sd:>7.4f} {dstr:>18s}   {flag}")

print(f"\n  Total wall time this session: {(time.time()-overall_t0)/60:.1f} min")
print(f"  Results JSON (resumable): {RESULTS_JSON}")
print(f"\n  Copy this entire output (and download the 100X JSON) and send it back.")


In [ ]:
import os, random, warnings, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve, accuracy_score, f1_score
warnings.filterwarnings('ignore')

# ╔══════════════════════════════════════════════════════════╗
# ║  CONFIG                                                   ║
# ╚══════════════════════════════════════════════════════════╝
class CFG:
    DATA_ROOT     = '../input/breakhis-400x-v1' 
    OUT_DIR       = '/kaggle/working'
    BATCH_SIZE    = 32
    FOCAL_GAMMA   = 2.0
    FOCAL_ALPHA   = 0.33          # BreaKHis benign:malignant ~1:2
    LABEL_SMOOTH  = 0.05
    SEEDS         = [42, 123, 2024]   # <-- 3 seeds; matches headline 200X protocol
    DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    AMP_ENABLED   = True
    TTA_FIVE_CROP = True
    # crude directional-flag thresholds (see docstring)
    FLAG_FLOOR    = 0.005         # ignore paired deltas smaller than half an AUC point

# Which magnifications THIS session runs. To split across sessions, set to ['40X'] or ['100X'].
MAGS_TO_RUN = ['200X']

# Per-magnification config (matches v0-10 headline hyperparameters)
MAG_CFG = {
    '40X':  dict(FREEZE_EPOCHS=15, UNFREEZE_EPOCHS=80,  DROPOUT_1=0.5, PATIENCE=35,
                 LR_P1=3e-4, LR_HEAD_P2=1e-4, LR_BACKBONE_P2=1e-5,
                 WEIGHT_DECAY=5e-3, P2_WARMUP=3, IMG_SIZE=224, AUG_STRENGTH='normal'),
    '100X': dict(FREEZE_EPOCHS=30, UNFREEZE_EPOCHS=90,  DROPOUT_1=0.5, PATIENCE=35,
                 LR_P1=3e-4, LR_HEAD_P2=1e-4, LR_BACKBONE_P2=1e-5,
                 WEIGHT_DECAY=5e-3, P2_WARMUP=3, IMG_SIZE=224, AUG_STRENGTH='normal'),
    '200X': dict(FREEZE_EPOCHS=12, UNFREEZE_EPOCHS=80,  DROPOUT_1=0.6, PATIENCE=35,
                 LR_P1=2e-4, LR_HEAD_P2=8e-5, LR_BACKBONE_P2=8e-6,
                 WEIGHT_DECAY=5e-3, P2_WARMUP=3, IMG_SIZE=224, AUG_STRENGTH='normal'),
    '400X': dict(FREEZE_EPOCHS=25, UNFREEZE_EPOCHS=100, DROPOUT_1=0.5, PATIENCE=15,
                 LR_P1=3e-4, LR_HEAD_P2=8e-5, LR_BACKBONE_P2=8e-6,
                 WEIGHT_DECAY=6e-3, P2_WARMUP=3, IMG_SIZE=320, AUG_STRENGTH='strong'),
}

ABLATIONS = ['full', 'no_mshra', 'no_spatial', 'no_channel', 'no_focal', 'no_twophase']

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.benchmark = True
os.makedirs(CFG.OUT_DIR, exist_ok=True)
RESULTS_JSON = os.path.join(CFG.OUT_DIR, 'ablation_200X_multiseed_results.json')
print(f"Device: {CFG.DEVICE}  |  Mags this session: {MAGS_TO_RUN}  |  Seeds: {CFG.SEEDS}")

# ╔══════════════════════════════════════════════════════════╗
# ║  RESUMABLE RESULT STORE                                   ║
# ║  key = f"{mag}|{seed}|{config}"  ->  metrics dict         ║
# ╚══════════════════════════════════════════════════════════╝
def load_store():
    if os.path.exists(RESULTS_JSON):
        with open(RESULTS_JSON) as f: return json.load(f)
    return {}
def save_store(store):
    tmp = RESULTS_JSON + '.tmp'
    with open(tmp, 'w') as f: json.dump(store, f, indent=2)
    os.replace(tmp, RESULTS_JSON)
STORE = load_store()
def skey(mag, seed, cfg): return f"{mag}|{seed}|{cfg}"

# ╔══════════════════════════════════════════════════════════╗
# ║  LOSS                                                     ║
# ╚══════════════════════════════════════════════════════════╝
def focal_loss(logits, labels):
    smooth = labels * (1 - CFG.LABEL_SMOOTH) + 0.5 * CFG.LABEL_SMOOTH
    bce = F.binary_cross_entropy_with_logits(logits, smooth, reduction='none')
    pt  = torch.where(labels == 1, torch.sigmoid(logits), 1 - torch.sigmoid(logits))
    a   = torch.where(labels == 1, torch.full_like(labels, CFG.FOCAL_ALPHA),
                      torch.full_like(labels, 1 - CFG.FOCAL_ALPHA))
    return (a * (1 - pt) ** CFG.FOCAL_GAMMA * bce).mean()

def plain_bce(logits, labels):
    return F.binary_cross_entropy_with_logits(logits, labels)

# ╔══════════════════════════════════════════════════════════╗
# ║  ATTENTION MODULES                                        ║
# ╚══════════════════════════════════════════════════════════╝
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False), nn.ReLU(inplace=True),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class MS_HRA_Block(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, use_channel=True, use_spatial=True):
        super().__init__()
        self.use_channel = use_channel
        self.use_spatial = use_spatial
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.ca    = ChannelAttention(out_ch) if use_channel else None
        self.sa    = SpatialAttention()       if use_spatial else None
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), nn.BatchNorm2d(out_ch))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.ca is not None: out = out * self.ca(out)
        if self.sa is not None: out = out * self.sa(out)
        return F.relu(out + self.shortcut(x))

# ╔══════════════════════════════════════════════════════════╗
# ║  MODEL (with ablation switch)                             ║
# ╚══════════════════════════════════════════════════════════╝
class Res_MSHRA(nn.Module):
    def __init__(self, dropout_1=0.5, ablation='full'):
        super().__init__()
        self.ablation = ablation
        base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.layer0 = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        if ablation == 'no_mshra':
            self.feature_stage = base.layer4   # genuine ResNet50 layer4: 1024->2048, stride 2
            self.is_mshra = False
        else:
            use_channel = (ablation != 'no_channel')
            use_spatial = (ablation != 'no_spatial')
            self.feature_stage = nn.Sequential(
                MS_HRA_Block(1024, 1024, stride=1, use_channel=use_channel, use_spatial=use_spatial),
                MS_HRA_Block(1024, 2048, stride=2, use_channel=use_channel, use_spatial=use_spatial))
            self.is_mshra = True
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(), nn.BatchNorm1d(2048), nn.Dropout(dropout_1),
            nn.Linear(2048, 512), nn.ReLU(inplace=True),
            nn.BatchNorm1d(512), nn.Dropout(0.3), nn.Linear(512, 1))
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.constant_(m.bias, 0)

    def backbone_params(self):
        p = (list(self.layer0.parameters()) + list(self.layer1.parameters()) +
             list(self.layer2.parameters()) + list(self.layer3.parameters()))
        if not self.is_mshra:
            p += list(self.feature_stage.parameters())
        return p
    def new_params(self):
        if self.is_mshra:
            return list(self.feature_stage.parameters()) + list(self.head.parameters())
        return list(self.head.parameters())
    def freeze_backbone(self):
        for p in self.backbone_params(): p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone_params(): p.requires_grad = True
    def forward(self, x):
        x = self.layer0(x); x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        x = self.feature_stage(x); x = self.gap(x)
        return self.head(x)

# ╔══════════════════════════════════════════════════════════╗
# ║  DATA                                                     ║
# ╚══════════════════════════════════════════════════════════╝
imagenet_norm = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

def get_train_transform(img_size, strength='normal'):
    if strength == 'strong':
        cj=(0.4,0.4,0.3,0.1); gray=0.10; erase_p=0.25; erase_sc=(0.02,0.15)
    else:
        cj=(0.2,0.2,0.15,0.05); gray=0.05; erase_p=0.15; erase_sc=(0.02,0.08)
    return transforms.Compose([
        transforms.Resize((int(img_size*1.15), int(img_size*1.15))),
        transforms.RandomCrop(img_size),
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=cj[0], contrast=cj[1], saturation=cj[2], hue=cj[3]),
        transforms.RandomGrayscale(p=gray),
        transforms.ToTensor(), imagenet_norm,
        transforms.RandomErasing(p=erase_p, scale=erase_sc)])

def get_val_transform(img_size):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)), transforms.ToTensor(), imagenet_norm])

class HistologyDataset(Dataset):
    def __init__(self, mag, split, transform=None):
        self.samples = []; self.transform = transform
        data_root = CFG.DATA_ROOT
        for root, dirs, _ in os.walk(data_root):
            if mag in dirs:
                data_root = root; break
        for cls, lbl in [('benign', 0), ('malignant', 1)]:
            path = os.path.join(data_root, mag, split, cls)
            if os.path.exists(path):
                for f in sorted(os.listdir(path)):
                    if f.lower().endswith(('.png','.jpg','.jpeg')):
                        self.samples.append((os.path.join(path, f), lbl))
        n_b = sum(1 for _,l in self.samples if l==0)
        n_m = sum(1 for _,l in self.samples if l==1)
        print(f"    [{mag}/{split}] benign={n_b} malignant={n_m} total={len(self.samples)}")
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, lbl = self.samples[idx]
        img = Image.open(path).convert('RGB')
        t = self.transform(img) if self.transform else transforms.ToTensor()(img)
        return t, torch.tensor(lbl, dtype=torch.float32)
    def get_labels(self): return [l for _,l in self.samples]

def make_train_loader(ds, seed):
    labels = np.array(ds.get_labels())
    counts = np.maximum(np.bincount(labels.astype(int), minlength=2), 1)
    w = (1.0 / counts)[labels.astype(int)]
    g = torch.Generator(); g.manual_seed(seed)   # seed the sampler too
    sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), len(ds),
                                    replacement=True, generator=g)
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                      num_workers=4, pin_memory=True, drop_last=True)

def make_val_loader(ds):
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                      num_workers=4, pin_memory=True, drop_last=False)

# ╔══════════════════════════════════════════════════════════╗
# ║  TRAIN / EVAL                                             ║
# ╚══════════════════════════════════════════════════════════╝
def train_one_epoch(model, loader, opt, scaler, loss_fn):
    model.train()
    for imgs, labels in loader:
        imgs=imgs.to(CFG.DEVICE,non_blocking=True); labels=labels.to(CFG.DEVICE,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        if CFG.AMP_ENABLED:
            with torch.cuda.amp.autocast():
                logits = model(imgs).squeeze(1); loss = loss_fn(logits, labels)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        else:
            logits = model(imgs).squeeze(1); loss = loss_fn(logits, labels)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()

@torch.no_grad()
def eval_auc(model, loader):
    model.eval(); probs=[]; tgts=[]
    for imgs, labels in loader:
        if CFG.AMP_ENABLED:
            with torch.cuda.amp.autocast():
                lo = model(imgs.to(CFG.DEVICE,non_blocking=True)).squeeze(1)
        else:
            lo = model(imgs.to(CFG.DEVICE,non_blocking=True)).squeeze(1)
        probs.extend(torch.sigmoid(lo.float()).cpu().numpy()); tgts.extend(labels.numpy())
    return roc_auc_score(tgts, probs)

@torch.no_grad()
def predict(model, loader, img_size, use_tta):
    model.eval(); P=[]; Y=[]
    crop = int(img_size*0.9)
    for imgs, labels in loader:
        imgs = imgs.to(CFG.DEVICE, non_blocking=True)
        if use_tta:
            views = [imgs, torch.flip(imgs,[3]), torch.flip(imgs,[2]), torch.flip(imgs,[2,3]),
                     torch.rot90(imgs,1,[2,3]), torch.rot90(imgs,3,[2,3])]
            if CFG.TTA_FIVE_CROP:
                H,W=imgs.shape[2],imgs.shape[3]
                offs=[((H-crop)//2,(W-crop)//2),(0,0),(0,W-crop),(H-crop,0),(H-crop,W-crop)]
                for t,l in offs:
                    c=imgs[:,:,t:t+crop,l:l+crop]
                    c=F.interpolate(c,size=(H,W),mode='bilinear',align_corners=False); views.append(c)
            if CFG.AMP_ENABLED:
                with torch.cuda.amp.autocast():
                    pr = sum(torch.sigmoid(model(v).float()) for v in views)/len(views)
            else:
                pr = sum(torch.sigmoid(model(v)) for v in views)/len(views)
        else:
            if CFG.AMP_ENABLED:
                with torch.cuda.amp.autocast():
                    pr = torch.sigmoid(model(imgs).float())
            else:
                pr = torch.sigmoid(model(imgs))
        P.extend(pr.squeeze(1).cpu().numpy()); Y.extend(labels.numpy())
    return np.array(P), np.array(Y)

def metrics_at_best_threshold(y_true, y_prob):
    fpr,tpr,thr = roc_curve(y_true,y_prob)
    accs=[accuracy_score(y_true,(y_prob>=t).astype(int)) for t in thr]
    bt=thr[int(np.argmax(accs))]
    y_pred=(y_prob>=bt).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_true,y_pred).ravel()
    sens=tp/(tp+fn) if (tp+fn)>0 else 0
    spec=tn/(tn+fp) if (tn+fp)>0 else 0
    return dict(auc=float(roc_auc_score(y_true,y_prob)), acc=float(max(accs)),
                sens=float(sens), spec=float(spec), f1=float(f1_score(y_true,y_pred)))

# ╔══════════════════════════════════════════════════════════╗
# ║  ONE (mag, seed, config) RUN                              ║
# ╚══════════════════════════════════════════════════════════╝
def run_one(mag, seed, ablation, train_loader, val_loader):
    mc = MAG_CFG[mag]; img_size = mc['IMG_SIZE']
    set_seed(seed)                              # per-seed, not fixed
    loss_fn = plain_bce if ablation == 'no_focal' else focal_loss
    two_phase = (ablation != 'no_twophase')

    model = Res_MSHRA(dropout_1=mc['DROPOUT_1'], ablation=ablation).to(CFG.DEVICE)
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.AMP_ENABLED)
    best_auc = 0.0
    ckpt = os.path.join(CFG.OUT_DIR, f'abl_{mag}_s{seed}_{ablation}.pth')

    if two_phase:
        model.freeze_backbone()
        opt = torch.optim.AdamW(model.new_params(), lr=mc['LR_P1'], weight_decay=mc['WEIGHT_DECAY'])
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=mc['FREEZE_EPOCHS'], eta_min=1e-6)
        for ep in range(1, mc['FREEZE_EPOCHS']+1):
            train_one_epoch(model, train_loader, opt, scaler, loss_fn)
            va = eval_auc(model, val_loader); sched.step()
            if va > best_auc: best_auc=va; torch.save(model.state_dict(), ckpt)
        model.unfreeze_backbone()
        opt = torch.optim.AdamW([
            {'params': model.backbone_params(), 'lr': mc['LR_BACKBONE_P2']},
            {'params': model.new_params(),      'lr': mc['LR_HEAD_P2']}],
            weight_decay=mc['WEIGHT_DECAY'])
        warm=mc['P2_WARMUP']; cos=mc['UNFREEZE_EPOCHS']-warm
        sched = torch.optim.lr_scheduler.SequentialLR(opt, schedulers=[
            torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=warm),
            torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cos, eta_min=1e-7)],
            milestones=[warm])
        patience=mc['PATIENCE']; ctr=0
        for ep in range(1, mc['UNFREEZE_EPOCHS']+1):
            train_one_epoch(model, train_loader, opt, scaler, loss_fn)
            va = eval_auc(model, val_loader); sched.step()
            if va > best_auc: best_auc=va; ctr=0; torch.save(model.state_dict(), ckpt)
            else: ctr += 1
            if ctr >= patience: break
    else:
        model.unfreeze_backbone()
        total_ep = mc['FREEZE_EPOCHS'] + mc['UNFREEZE_EPOCHS']
        opt = torch.optim.AdamW([
            {'params': model.backbone_params(), 'lr': mc['LR_BACKBONE_P2']},
            {'params': model.new_params(),      'lr': mc['LR_HEAD_P2']}],
            weight_decay=mc['WEIGHT_DECAY'])
        warm=mc['P2_WARMUP']; cos=total_ep-warm
        sched = torch.optim.lr_scheduler.SequentialLR(opt, schedulers=[
            torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=warm),
            torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cos, eta_min=1e-7)],
            milestones=[warm])
        patience=mc['PATIENCE']; ctr=0
        for ep in range(1, total_ep+1):
            train_one_epoch(model, train_loader, opt, scaler, loss_fn)
            va = eval_auc(model, val_loader); sched.step()
            if va > best_auc: best_auc=va; ctr=0; torch.save(model.state_dict(), ckpt)
            else: ctr += 1
            if ctr >= patience: break

    model.load_state_dict(torch.load(ckpt, map_location=CFG.DEVICE))
    yP, yT = predict(model, val_loader, img_size, use_tta=True)
    m_tta = metrics_at_best_threshold(yT, yP)
    m_notta = None
    if ablation == 'full':
        yP2, yT2 = predict(model, val_loader, img_size, use_tta=False)
        m_notta = metrics_at_best_threshold(yT2, yP2)
    try: os.remove(ckpt)                         # keep /kaggle/working small
    except OSError: pass
    return m_tta, m_notta

# ╔══════════════════════════════════════════════════════════╗
# ║  RUN: loop mag -> seed -> config (resumable)              ║
# ╚══════════════════════════════════════════════════════════╝
overall_t0 = time.time()
for mag in MAGS_TO_RUN:
    print(f"\n{'='*64}\n  MAGNIFICATION {mag}\n{'='*64}")
    mc = MAG_CFG[mag]; img_size = mc['IMG_SIZE']
    for seed in CFG.SEEDS:
        print(f"\n  ----- {mag} | seed {seed} -----")
        tr_ds = HistologyDataset(mag, 'train', get_train_transform(img_size, mc['AUG_STRENGTH']))
        va_ds = HistologyDataset(mag, 'test',  get_val_transform(img_size))
        tr_loader = make_train_loader(tr_ds, seed); va_loader = make_val_loader(va_ds)
        for ab in ABLATIONS:
            if skey(mag, seed, ab) in STORE and (ab != 'full' or skey(mag, seed, 'full_no_tta') in STORE):
                m = STORE[skey(mag, seed, ab)]
                print(f"    [skip] {ab:12s} AUC={m['auc']:.4f} (cached)")
                continue
            t0 = time.time()
            m_tta, m_notta = run_one(mag, seed, ab, tr_loader, va_loader)
            STORE[skey(mag, seed, ab)] = m_tta
            if ab == 'full' and m_notta is not None:
                STORE[skey(mag, seed, 'full_no_tta')] = m_notta
            save_store(STORE)
            print(f"    {ab:12s} AUC={m_tta['auc']:.4f} Acc={m_tta['acc']*100:.2f}% "
                  f"Sens={m_tta['sens']:.3f} Spec={m_tta['spec']:.3f} F1={m_tta['f1']:.3f}  "
                  f"({(time.time()-t0)/60:.1f} min)")
            if ab == 'full' and m_notta is not None:
                print(f"    {'full(noTTA)':12s} AUC={m_notta['auc']:.4f} (free)")

# ╔══════════════════════════════════════════════════════════╗
# ║  AGGREGATE: mean +/- std + PAIRED dAUC across seeds       ║
# ╚══════════════════════════════════════════════════════════╝
ROW_ORDER = ['full', 'full_no_tta', 'no_mshra', 'no_spatial', 'no_channel', 'no_focal', 'no_twophase']
ROW_LABEL = {
    'full':        'Full Res-MSHRA',
    'full_no_tta': '  - TTA (full model, no TTA)',
    'no_mshra':    '  - MS-HRA blocks (plain ResNet50)',
    'no_spatial':  '  - Spatial attention',
    'no_channel':  '  - Channel attention',
    'no_focal':    '  - Focal loss (plain BCE)',
    'no_twophase': '  - Two-phase training (end-to-end)',
}

def collect(mag, cfg, metric='auc'):
    vals=[]
    for seed in CFG.SEEDS:
        k=skey(mag,seed,cfg)
        if k in STORE: vals.append(STORE[k][metric])
    return np.array(vals, dtype=float)

print(f"\n\n{'#'*82}")
print(f"  MULTI-SEED ABLATION — 200X only  seeds={CFG.SEEDS}")
print(f"  paired dAUC = mean_seed( AUC(config) - AUC(full) ).  negative => component HELPED.")
print(f"  flag '*' = |mean dAUC| >= std dAUC AND |mean dAUC| >= {CFG.FLAG_FLOOR} (crude, not a p-value)")
print(f"{'#'*82}")

for mag in MAGS_TO_RUN:
    full_aucs = collect(mag, 'full')
    if len(full_aucs)==0:
        print(f"\n=== {mag} === (no results yet)"); continue
    n = len(full_aucs)
    print(f"\n=== {mag} ===   (n={n} seeds; full AUC = {full_aucs.mean():.4f} +/- {full_aucs.std(ddof=1) if n>1 else 0:.4f})")
    print(f"  {'Configuration':<36s} {'mean AUC':>10s} {'std':>7s} {'paired dAUC':>18s}  flag")
    print(f"  {'-'*80}")
    for key in ROW_ORDER:
        aucs = collect(mag, key)
        if len(aucs)==0: continue
        mu = aucs.mean(); sd = aucs.std(ddof=1) if len(aucs)>1 else 0.0
        if key == 'full':
            print(f"  {ROW_LABEL[key]:<36s} {mu:>10.4f} {sd:>7.4f} {'(reference)':>18s}")
            continue
        # paired delta: align seed-by-seed against full (or full_no_tta's own full)
        deltas=[]
        for seed in CFG.SEEDS:
            kc=skey(mag,seed,key); kf=skey(mag,seed,'full')
            if kc in STORE and kf in STORE:
                deltas.append(STORE[kc]['auc'] - STORE[kf]['auc'])
        deltas=np.array(deltas, dtype=float)
        if len(deltas)>0:
            dmu=deltas.mean(); dsd=deltas.std(ddof=1) if len(deltas)>1 else 0.0
            flag = '*' if (abs(dmu) >= dsd and abs(dmu) >= CFG.FLAG_FLOOR) else ''
            dstr=f"{dmu:+.4f} +/- {dsd:.4f}"
        else:
            dstr="n/a"; flag=''
        print(f"  {ROW_LABEL[key]:<36s} {mu:>10.4f} {sd:>7.4f} {dstr:>18s}   {flag}")

print(f"\n  Total wall time this session: {(time.time()-overall_t0)/60:.1f} min")
print(f"  Results JSON (resumable): {RESULTS_JSON}")
print(f"\n  Copy this entire output (and download the 200X JSON) and send it back.")


In [ ]:
import os, random, warnings, time, json
import numpy as np
import torch
import torch._utils  # force-load lazy submodule: avoids intermittent
                     # "module 'torch' has no attribute '_utils'" during
                     # ResNet50 weight deserialization on Kaggle Torch 2.x
import torch.serialization
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve, accuracy_score, f1_score
warnings.filterwarnings('ignore')

# ╔══════════════════════════════════════════════════════════╗
# ║  CONFIG                                                   ║
# ╚══════════════════════════════════════════════════════════╝
class CFG:
    DATA_ROOT     = '../input/breakhis-dataset/BreaKHis_v1'
    OUT_DIR       = '/kaggle/working'
    BATCH_SIZE    = 32
    FOCAL_GAMMA   = 2.0
    FOCAL_ALPHA   = 0.33          # BreaKHis benign:malignant ~1:2
    LABEL_SMOOTH  = 0.05
    SEEDS         = [42, 123, 2024, 7, 1337]   # <-- 5 seeds; matches headline 400X protocol
    DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    AMP_ENABLED   = True
    TTA_FIVE_CROP = True
    # crude directional-flag thresholds (see docstring)
    FLAG_FLOOR    = 0.005         # ignore paired deltas smaller than half an AUC point

# Which magnifications THIS session runs. To split across sessions, set to ['40X'] or ['100X'].
MAGS_TO_RUN = ['400X']

# Per-magnification config (matches v0-10 headline hyperparameters)
MAG_CFG = {
    '40X':  dict(FREEZE_EPOCHS=15, UNFREEZE_EPOCHS=80,  DROPOUT_1=0.5, PATIENCE=35,
                 LR_P1=3e-4, LR_HEAD_P2=1e-4, LR_BACKBONE_P2=1e-5,
                 WEIGHT_DECAY=5e-3, P2_WARMUP=3, IMG_SIZE=224, AUG_STRENGTH='normal'),
    '100X': dict(FREEZE_EPOCHS=30, UNFREEZE_EPOCHS=90,  DROPOUT_1=0.5, PATIENCE=35,
                 LR_P1=3e-4, LR_HEAD_P2=1e-4, LR_BACKBONE_P2=1e-5,
                 WEIGHT_DECAY=5e-3, P2_WARMUP=3, IMG_SIZE=224, AUG_STRENGTH='normal'),
    '200X': dict(FREEZE_EPOCHS=12, UNFREEZE_EPOCHS=80,  DROPOUT_1=0.6, PATIENCE=35,
                 LR_P1=2e-4, LR_HEAD_P2=8e-5, LR_BACKBONE_P2=8e-6,
                 WEIGHT_DECAY=5e-3, P2_WARMUP=3, IMG_SIZE=224, AUG_STRENGTH='normal'),
    '400X': dict(FREEZE_EPOCHS=25, UNFREEZE_EPOCHS=100, DROPOUT_1=0.5, PATIENCE=15,
                 LR_P1=3e-4, LR_HEAD_P2=8e-5, LR_BACKBONE_P2=8e-6,
                 WEIGHT_DECAY=6e-3, P2_WARMUP=3, IMG_SIZE=320, AUG_STRENGTH='strong'),
}

ABLATIONS = ['full', 'no_mshra', 'no_spatial', 'no_channel', 'no_focal', 'no_twophase']

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.benchmark = True
os.makedirs(CFG.OUT_DIR, exist_ok=True)
RESULTS_JSON = os.path.join(CFG.OUT_DIR, 'ablation_400X_multiseed_results.json')
print(f"Device: {CFG.DEVICE}  |  Mags this session: {MAGS_TO_RUN}  |  Seeds: {CFG.SEEDS}")

# ╔══════════════════════════════════════════════════════════╗
# ║  RESUMABLE RESULT STORE                                   ║
# ║  key = f"{mag}|{seed}|{config}"  ->  metrics dict         ║
# ╚══════════════════════════════════════════════════════════╝
def load_store():
    if os.path.exists(RESULTS_JSON):
        with open(RESULTS_JSON) as f: return json.load(f)
    return {}
def save_store(store):
    tmp = RESULTS_JSON + '.tmp'
    with open(tmp, 'w') as f: json.dump(store, f, indent=2)
    os.replace(tmp, RESULTS_JSON)
STORE = load_store()
def skey(mag, seed, cfg): return f"{mag}|{seed}|{cfg}"

# ╔══════════════════════════════════════════════════════════╗
# ║  LOSS                                                     ║
# ╚══════════════════════════════════════════════════════════╝
def focal_loss(logits, labels):
    smooth = labels * (1 - CFG.LABEL_SMOOTH) + 0.5 * CFG.LABEL_SMOOTH
    bce = F.binary_cross_entropy_with_logits(logits, smooth, reduction='none')
    pt  = torch.where(labels == 1, torch.sigmoid(logits), 1 - torch.sigmoid(logits))
    a   = torch.where(labels == 1, torch.full_like(labels, CFG.FOCAL_ALPHA),
                      torch.full_like(labels, 1 - CFG.FOCAL_ALPHA))
    return (a * (1 - pt) ** CFG.FOCAL_GAMMA * bce).mean()

def plain_bce(logits, labels):
    return F.binary_cross_entropy_with_logits(logits, labels)

# ╔══════════════════════════════════════════════════════════╗
# ║  ATTENTION MODULES                                        ║
# ╚══════════════════════════════════════════════════════════╝
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False), nn.ReLU(inplace=True),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class MS_HRA_Block(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, use_channel=True, use_spatial=True):
        super().__init__()
        self.use_channel = use_channel
        self.use_spatial = use_spatial
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.ca    = ChannelAttention(out_ch) if use_channel else None
        self.sa    = SpatialAttention()       if use_spatial else None
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), nn.BatchNorm2d(out_ch))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.ca is not None: out = out * self.ca(out)
        if self.sa is not None: out = out * self.sa(out)
        return F.relu(out + self.shortcut(x))

# ╔══════════════════════════════════════════════════════════╗
# ║  MODEL (with ablation switch)                             ║
# ╚══════════════════════════════════════════════════════════╝
def _load_resnet50_safely():
    """Load ResNet50 IMAGENET1K_V2 weights, retrying once if the cached file is
    corrupt (can happen if a prior run died mid-download). Pure environment
    robustness; does not alter the model."""
    import glob, torch.hub
    try:
        return models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    except Exception as e:
        print(f"    [warn] resnet50 load failed ({type(e).__name__}: {e}); clearing cache + retry")
        cache_dir = os.path.join(torch.hub.get_dir(), 'checkpoints')
        for f in glob.glob(os.path.join(cache_dir, 'resnet50-*.pth')):
            try: os.remove(f)
            except OSError: pass
        return models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)


class Res_MSHRA(nn.Module):
    def __init__(self, dropout_1=0.5, ablation='full'):
        super().__init__()
        self.ablation = ablation
        base = _load_resnet50_safely()
        self.layer0 = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        if ablation == 'no_mshra':
            self.feature_stage = base.layer4   # genuine ResNet50 layer4: 1024->2048, stride 2
            self.is_mshra = False
        else:
            use_channel = (ablation != 'no_channel')
            use_spatial = (ablation != 'no_spatial')
            self.feature_stage = nn.Sequential(
                MS_HRA_Block(1024, 1024, stride=1, use_channel=use_channel, use_spatial=use_spatial),
                MS_HRA_Block(1024, 2048, stride=2, use_channel=use_channel, use_spatial=use_spatial))
            self.is_mshra = True
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(), nn.BatchNorm1d(2048), nn.Dropout(dropout_1),
            nn.Linear(2048, 512), nn.ReLU(inplace=True),
            nn.BatchNorm1d(512), nn.Dropout(0.3), nn.Linear(512, 1))
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.constant_(m.bias, 0)

    def backbone_params(self):
        p = (list(self.layer0.parameters()) + list(self.layer1.parameters()) +
             list(self.layer2.parameters()) + list(self.layer3.parameters()))
        if not self.is_mshra:
            p += list(self.feature_stage.parameters())
        return p
    def new_params(self):
        if self.is_mshra:
            return list(self.feature_stage.parameters()) + list(self.head.parameters())
        return list(self.head.parameters())
    def freeze_backbone(self):
        for p in self.backbone_params(): p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone_params(): p.requires_grad = True
    def forward(self, x):
        x = self.layer0(x); x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        x = self.feature_stage(x); x = self.gap(x)
        return self.head(x)

# ╔══════════════════════════════════════════════════════════╗
# ║  DATA                                                     ║
# ╚══════════════════════════════════════════════════════════╝
imagenet_norm = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

def get_train_transform(img_size, strength='normal'):
    if strength == 'strong':
        cj=(0.4,0.4,0.3,0.1); gray=0.10; erase_p=0.25; erase_sc=(0.02,0.15)
    else:
        cj=(0.2,0.2,0.15,0.05); gray=0.05; erase_p=0.15; erase_sc=(0.02,0.08)
    return transforms.Compose([
        transforms.Resize((int(img_size*1.15), int(img_size*1.15))),
        transforms.RandomCrop(img_size),
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=cj[0], contrast=cj[1], saturation=cj[2], hue=cj[3]),
        transforms.RandomGrayscale(p=gray),
        transforms.ToTensor(), imagenet_norm,
        transforms.RandomErasing(p=erase_p, scale=erase_sc)])

def get_val_transform(img_size):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)), transforms.ToTensor(), imagenet_norm])

class HistologyDataset(Dataset):
    def __init__(self, mag, split, transform=None):
        self.samples = []; self.transform = transform
        data_root = CFG.DATA_ROOT
        for root, dirs, _ in os.walk(data_root):
            if mag in dirs:
                data_root = root; break
        for cls, lbl in [('benign', 0), ('malignant', 1)]:
            path = os.path.join(data_root, mag, split, cls)
            if os.path.exists(path):
                for f in sorted(os.listdir(path)):
                    if f.lower().endswith(('.png','.jpg','.jpeg')):
                        self.samples.append((os.path.join(path, f), lbl))
        n_b = sum(1 for _,l in self.samples if l==0)
        n_m = sum(1 for _,l in self.samples if l==1)
        print(f"    [{mag}/{split}] benign={n_b} malignant={n_m} total={len(self.samples)}")
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, lbl = self.samples[idx]
        img = Image.open(path).convert('RGB')
        t = self.transform(img) if self.transform else transforms.ToTensor()(img)
        return t, torch.tensor(lbl, dtype=torch.float32)
    def get_labels(self): return [l for _,l in self.samples]

def make_train_loader(ds, seed):
    labels = np.array(ds.get_labels())
    counts = np.maximum(np.bincount(labels.astype(int), minlength=2), 1)
    w = (1.0 / counts)[labels.astype(int)]
    g = torch.Generator(); g.manual_seed(seed)   # seed the sampler too
    sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), len(ds),
                                    replacement=True, generator=g)
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                      num_workers=4, pin_memory=True, drop_last=True)

def make_val_loader(ds):
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                      num_workers=4, pin_memory=True, drop_last=False)

# ╔══════════════════════════════════════════════════════════╗
# ║  TRAIN / EVAL                                             ║
# ╚══════════════════════════════════════════════════════════╝
def train_one_epoch(model, loader, opt, scaler, loss_fn):
    model.train()
    for imgs, labels in loader:
        imgs=imgs.to(CFG.DEVICE,non_blocking=True); labels=labels.to(CFG.DEVICE,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        if CFG.AMP_ENABLED:
            with torch.cuda.amp.autocast():
                logits = model(imgs).squeeze(1); loss = loss_fn(logits, labels)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        else:
            logits = model(imgs).squeeze(1); loss = loss_fn(logits, labels)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()

@torch.no_grad()
def eval_auc(model, loader):
    model.eval(); probs=[]; tgts=[]
    for imgs, labels in loader:
        if CFG.AMP_ENABLED:
            with torch.cuda.amp.autocast():
                lo = model(imgs.to(CFG.DEVICE,non_blocking=True)).squeeze(1)
        else:
            lo = model(imgs.to(CFG.DEVICE,non_blocking=True)).squeeze(1)
        probs.extend(torch.sigmoid(lo.float()).cpu().numpy()); tgts.extend(labels.numpy())
    return roc_auc_score(tgts, probs)

@torch.no_grad()
def predict(model, loader, img_size, use_tta):
    model.eval(); P=[]; Y=[]
    crop = int(img_size*0.9)
    for imgs, labels in loader:
        imgs = imgs.to(CFG.DEVICE, non_blocking=True)
        if use_tta:
            views = [imgs, torch.flip(imgs,[3]), torch.flip(imgs,[2]), torch.flip(imgs,[2,3]),
                     torch.rot90(imgs,1,[2,3]), torch.rot90(imgs,3,[2,3])]
            if CFG.TTA_FIVE_CROP:
                H,W=imgs.shape[2],imgs.shape[3]
                offs=[((H-crop)//2,(W-crop)//2),(0,0),(0,W-crop),(H-crop,0),(H-crop,W-crop)]
                for t,l in offs:
                    c=imgs[:,:,t:t+crop,l:l+crop]
                    c=F.interpolate(c,size=(H,W),mode='bilinear',align_corners=False); views.append(c)
            if CFG.AMP_ENABLED:
                with torch.cuda.amp.autocast():
                    pr = sum(torch.sigmoid(model(v).float()) for v in views)/len(views)
            else:
                pr = sum(torch.sigmoid(model(v)) for v in views)/len(views)
        else:
            if CFG.AMP_ENABLED:
                with torch.cuda.amp.autocast():
                    pr = torch.sigmoid(model(imgs).float())
            else:
                pr = torch.sigmoid(model(imgs))
        P.extend(pr.squeeze(1).cpu().numpy()); Y.extend(labels.numpy())
    return np.array(P), np.array(Y)

def metrics_at_best_threshold(y_true, y_prob):
    fpr,tpr,thr = roc_curve(y_true,y_prob)
    accs=[accuracy_score(y_true,(y_prob>=t).astype(int)) for t in thr]
    bt=thr[int(np.argmax(accs))]
    y_pred=(y_prob>=bt).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_true,y_pred).ravel()
    sens=tp/(tp+fn) if (tp+fn)>0 else 0
    spec=tn/(tn+fp) if (tn+fp)>0 else 0
    return dict(auc=float(roc_auc_score(y_true,y_prob)), acc=float(max(accs)),
                sens=float(sens), spec=float(spec), f1=float(f1_score(y_true,y_pred)))

# ╔══════════════════════════════════════════════════════════╗
# ║  ONE (mag, seed, config) RUN                              ║
# ╚══════════════════════════════════════════════════════════╝
def run_one(mag, seed, ablation, train_loader, val_loader):
    mc = MAG_CFG[mag]; img_size = mc['IMG_SIZE']
    set_seed(seed)                              # per-seed, not fixed
    loss_fn = plain_bce if ablation == 'no_focal' else focal_loss
    two_phase = (ablation != 'no_twophase')

    model = Res_MSHRA(dropout_1=mc['DROPOUT_1'], ablation=ablation).to(CFG.DEVICE)
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.AMP_ENABLED)
    best_auc = 0.0
    ckpt = os.path.join(CFG.OUT_DIR, f'abl_{mag}_s{seed}_{ablation}.pth')

    if two_phase:
        model.freeze_backbone()
        opt = torch.optim.AdamW(model.new_params(), lr=mc['LR_P1'], weight_decay=mc['WEIGHT_DECAY'])
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=mc['FREEZE_EPOCHS'], eta_min=1e-6)
        for ep in range(1, mc['FREEZE_EPOCHS']+1):
            train_one_epoch(model, train_loader, opt, scaler, loss_fn)
            va = eval_auc(model, val_loader); sched.step()
            if va > best_auc: best_auc=va; torch.save(model.state_dict(), ckpt)
        model.unfreeze_backbone()
        opt = torch.optim.AdamW([
            {'params': model.backbone_params(), 'lr': mc['LR_BACKBONE_P2']},
            {'params': model.new_params(),      'lr': mc['LR_HEAD_P2']}],
            weight_decay=mc['WEIGHT_DECAY'])
        warm=mc['P2_WARMUP']; cos=mc['UNFREEZE_EPOCHS']-warm
        sched = torch.optim.lr_scheduler.SequentialLR(opt, schedulers=[
            torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=warm),
            torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cos, eta_min=1e-7)],
            milestones=[warm])
        patience=mc['PATIENCE']; ctr=0
        for ep in range(1, mc['UNFREEZE_EPOCHS']+1):
            train_one_epoch(model, train_loader, opt, scaler, loss_fn)
            va = eval_auc(model, val_loader); sched.step()
            if va > best_auc: best_auc=va; ctr=0; torch.save(model.state_dict(), ckpt)
            else: ctr += 1
            if ctr >= patience: break
    else:
        model.unfreeze_backbone()
        total_ep = mc['FREEZE_EPOCHS'] + mc['UNFREEZE_EPOCHS']
        opt = torch.optim.AdamW([
            {'params': model.backbone_params(), 'lr': mc['LR_BACKBONE_P2']},
            {'params': model.new_params(),      'lr': mc['LR_HEAD_P2']}],
            weight_decay=mc['WEIGHT_DECAY'])
        warm=mc['P2_WARMUP']; cos=total_ep-warm
        sched = torch.optim.lr_scheduler.SequentialLR(opt, schedulers=[
            torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=warm),
            torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cos, eta_min=1e-7)],
            milestones=[warm])
        patience=mc['PATIENCE']; ctr=0
        for ep in range(1, total_ep+1):
            train_one_epoch(model, train_loader, opt, scaler, loss_fn)
            va = eval_auc(model, val_loader); sched.step()
            if va > best_auc: best_auc=va; ctr=0; torch.save(model.state_dict(), ckpt)
            else: ctr += 1
            if ctr >= patience: break

    model.load_state_dict(torch.load(ckpt, map_location=CFG.DEVICE))
    yP, yT = predict(model, val_loader, img_size, use_tta=True)
    m_tta = metrics_at_best_threshold(yT, yP)
    m_notta = None
    if ablation == 'full':
        yP2, yT2 = predict(model, val_loader, img_size, use_tta=False)
        m_notta = metrics_at_best_threshold(yT2, yP2)
    try: os.remove(ckpt)                         # keep /kaggle/working small
    except OSError: pass
    return m_tta, m_notta

# ╔══════════════════════════════════════════════════════════╗
# ║  RUN: loop mag -> seed -> config (resumable)              ║
# ╚══════════════════════════════════════════════════════════╝
overall_t0 = time.time()
for mag in MAGS_TO_RUN:
    print(f"\n{'='*64}\n  MAGNIFICATION {mag}\n{'='*64}")
    mc = MAG_CFG[mag]; img_size = mc['IMG_SIZE']
    for seed in CFG.SEEDS:
        print(f"\n  ----- {mag} | seed {seed} -----")
        tr_ds = HistologyDataset(mag, 'train', get_train_transform(img_size, mc['AUG_STRENGTH']))
        va_ds = HistologyDataset(mag, 'test',  get_val_transform(img_size))
        tr_loader = make_train_loader(tr_ds, seed); va_loader = make_val_loader(va_ds)
        for ab in ABLATIONS:
            if skey(mag, seed, ab) in STORE and (ab != 'full' or skey(mag, seed, 'full_no_tta') in STORE):
                m = STORE[skey(mag, seed, ab)]
                print(f"    [skip] {ab:12s} AUC={m['auc']:.4f} (cached)")
                continue
            t0 = time.time()
            m_tta, m_notta = run_one(mag, seed, ab, tr_loader, va_loader)
            STORE[skey(mag, seed, ab)] = m_tta
            if ab == 'full' and m_notta is not None:
                STORE[skey(mag, seed, 'full_no_tta')] = m_notta
            save_store(STORE)
            print(f"    {ab:12s} AUC={m_tta['auc']:.4f} Acc={m_tta['acc']*100:.2f}% "
                  f"Sens={m_tta['sens']:.3f} Spec={m_tta['spec']:.3f} F1={m_tta['f1']:.3f}  "
                  f"({(time.time()-t0)/60:.1f} min)")
            if ab == 'full' and m_notta is not None:
                print(f"    {'full(noTTA)':12s} AUC={m_notta['auc']:.4f} (free)")

# ╔══════════════════════════════════════════════════════════╗
# ║  AGGREGATE: mean +/- std + PAIRED dAUC across seeds       ║
# ╚══════════════════════════════════════════════════════════╝
ROW_ORDER = ['full', 'full_no_tta', 'no_mshra', 'no_spatial', 'no_channel', 'no_focal', 'no_twophase']
ROW_LABEL = {
    'full':        'Full Res-MSHRA',
    'full_no_tta': '  - TTA (full model, no TTA)',
    'no_mshra':    '  - MS-HRA blocks (plain ResNet50)',
    'no_spatial':  '  - Spatial attention',
    'no_channel':  '  - Channel attention',
    'no_focal':    '  - Focal loss (plain BCE)',
    'no_twophase': '  - Two-phase training (end-to-end)',
}

def collect(mag, cfg, metric='auc'):
    vals=[]
    for seed in CFG.SEEDS:
        k=skey(mag,seed,cfg)
        if k in STORE: vals.append(STORE[k][metric])
    return np.array(vals, dtype=float)

print(f"\n\n{'#'*82}")
print(f"  MULTI-SEED ABLATION — 400X only  seeds={CFG.SEEDS}")
print(f"  paired dAUC = mean_seed( AUC(config) - AUC(full) ).  negative => component HELPED.")
print(f"  flag '*' = |mean dAUC| >= std dAUC AND |mean dAUC| >= {CFG.FLAG_FLOOR} (crude, not a p-value)")
print(f"{'#'*82}")

for mag in MAGS_TO_RUN:
    full_aucs = collect(mag, 'full')
    if len(full_aucs)==0:
        print(f"\n=== {mag} === (no results yet)"); continue
    n = len(full_aucs)
    print(f"\n=== {mag} ===   (n={n} seeds; full AUC = {full_aucs.mean():.4f} +/- {full_aucs.std(ddof=1) if n>1 else 0:.4f})")
    print(f"  {'Configuration':<36s} {'mean AUC':>10s} {'std':>7s} {'paired dAUC':>18s}  flag")
    print(f"  {'-'*80}")
    for key in ROW_ORDER:
        aucs = collect(mag, key)
        if len(aucs)==0: continue
        mu = aucs.mean(); sd = aucs.std(ddof=1) if len(aucs)>1 else 0.0
        if key == 'full':
            print(f"  {ROW_LABEL[key]:<36s} {mu:>10.4f} {sd:>7.4f} {'(reference)':>18s}")
            continue
        # paired delta: align seed-by-seed against full (or full_no_tta's own full)
        deltas=[]
        for seed in CFG.SEEDS:
            kc=skey(mag,seed,key); kf=skey(mag,seed,'full')
            if kc in STORE and kf in STORE:
                deltas.append(STORE[kc]['auc'] - STORE[kf]['auc'])
        deltas=np.array(deltas, dtype=float)
        if len(deltas)>0:
            dmu=deltas.mean(); dsd=deltas.std(ddof=1) if len(deltas)>1 else 0.0
            flag = '*' if (abs(dmu) >= dsd and abs(dmu) >= CFG.FLAG_FLOOR) else ''
            dstr=f"{dmu:+.4f} +/- {dsd:.4f}"
        else:
            dstr="n/a"; flag=''
        print(f"  {ROW_LABEL[key]:<36s} {mu:>10.4f} {sd:>7.4f} {dstr:>18s}   {flag}")

print(f"\n  Total wall time this session: {(time.time()-overall_t0)/60:.1f} min")
print(f"  Results JSON (resumable): {RESULTS_JSON}")
print(f"\n  Copy this entire output (and download the 400X JSON) and send it back.")
